# Brazilian E-Commerce (Olist) — ETL para Presentación Ejecutiva (Plan v2)

Este notebook lee los datasets crudos de Olist desde `public/dataset/`, calcula todos los
agregados del **plan v2 de la presentación** y exporta archivos JSON ligeros a `src/data/`
para alimentar los componentes de React (Next.js + Reveal.js + Recharts).

**Objetivo del análisis:** cuantificar el impacto financiero de las reseñas negativas
(1–2★) cruzando `payment_value`, tiempos de cumplimiento y `freight_value`.

**Historias de usuario → archivos generados:**

| Slide | Historia | Archivo |
|---|---|---|
| 2 | Data Product Canvas (evidencia) | `kpis.json` |
| 3 | Historia 1 — Impacto de la latencia (logística) | `latency_review.json` |
| 4 | Historia 2 — Cuantificando el riesgo (financiero) | `payment_by_score.json`, `financial_kpis.json` |
| 5 | Historia 3 — Culpables y oportunidades (comercial) | `category_matrix.json` |


In [1]:
import json
from pathlib import Path

import pandas as pd

DATASET = Path("public/dataset")
OUT = Path("src/data")
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


## 1. Carga de datos

In [2]:
orders = pd.read_csv(
    DATASET / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
items = pd.read_csv(DATASET / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATASET / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(
    DATASET / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
customers = pd.read_csv(DATASET / "olist_customers_dataset.csv")
products = pd.read_csv(DATASET / "olist_products_dataset.csv")
sellers = pd.read_csv(DATASET / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATASET / "product_category_name_translation.csv")
# El encabezado de la traducción trae BOM
category_translation.columns = [c.lstrip("\ufeff") for c in category_translation.columns]

print(f"orders: {len(orders):,} | items: {len(items):,} | payments: {len(payments):,}")
print(f"reviews: {len(reviews):,} | customers: {len(customers):,} | products: {len(products):,}")


orders: 99,441 | items: 112,650 | payments: 103,886
reviews: 99,224 | customers: 99,441 | products: 32,951


## 2. Preparación

- Una reseña por orden (la más reciente, si hay varias).
- Pago total por orden.
- Región geográfica del cliente a partir de su estado (UF).
- `delay_days`: días de atraso (+) o anticipación (−) frente a la fecha estimada.


In [3]:
# Una resena por orden: conservamos la mas reciente
reviews_per_order = (
    reviews.sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
)

# Pago total por orden (una orden puede tener varios registros de pago)
payment_per_order = payments.groupby("order_id", as_index=False)["payment_value"].sum()

# Region de Brasil por estado del cliente
STATE_TO_REGION = {
    **dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
    **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
    **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
    **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
    **dict.fromkeys(["PR", "RS", "SC"], "Sul"),
}
customers["region"] = customers["customer_state"].map(STATE_TO_REGION)

orders_full = (
    orders.merge(payment_per_order, on="order_id", how="left")
    .merge(reviews_per_order, on="order_id", how="left")
    .merge(customers[["customer_id", "customer_state", "region"]], on="customer_id", how="left")
)

delivered = orders_full[orders_full["order_status"] == "delivered"].copy()
delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days

print(f"Ordenes totales: {len(orders_full):,} | entregadas: {len(delivered):,}")


Ordenes totales: 99,441 | entregadas: 96,478


## 3. KPIs generales y del Canvas → `kpis.json`

Métricas de contexto para la portada y el Data Product Canvas, incluido el riesgo
identificado en el plan: reseñas sin comentario (`review_comment_message` vacío),
que limitan el análisis cualitativo.


In [4]:
total_revenue = float(payments["payment_value"].sum())
total_orders = int(orders["order_id"].nunique())

# Ticket promedio: valor de la orden (productos + flete) promediado sobre ordenes con items
order_value = items.groupby("order_id").agg(total=("price", "sum"), freight=("freight_value", "sum"))
avg_ticket = float((order_value["total"] + order_value["freight"]).mean())

delivery_rate = float((orders["order_status"] == "delivered").mean() * 100)
avg_review = float(reviews_per_order["review_score"].mean())

late_mask = delivered["delay_days"] > 0
late_rate = float(late_mask.mean() * 100)
on_time_score = float(delivered.loc[~late_mask, "review_score"].mean())
late_score = float(delivered.loc[late_mask, "review_score"].mean())
late_revenue = float(delivered.loc[late_mask, "payment_value"].sum())

# Riesgo del Canvas: resenas sin comentario limitan el analisis cualitativo
reviews_without_comment = int(reviews["review_comment_message"].isna().sum())

kpis = {
    "totalRevenue": round(total_revenue, 2),
    "totalOrders": total_orders,
    "avgTicket": round(avg_ticket, 2),
    "deliveryRate": round(delivery_rate, 2),
    "avgReviewScore": round(avg_review, 2),
    "totalCustomers": int(customers["customer_unique_id"].nunique()),
    "totalSellers": int(sellers["seller_id"].nunique()),
    "lateDeliveryRate": round(late_rate, 2),
    "onTimeAvgScore": round(on_time_score, 2),
    "lateAvgScore": round(late_score, 2),
    "lateOrdersRevenue": round(late_revenue, 2),
    "reviewsWithoutComment": reviews_without_comment,
    "totalReviews": int(len(reviews_per_order)),
}
(OUT / "kpis.json").write_text(json.dumps(kpis, indent=2, ensure_ascii=False))
kpis


{'totalRevenue': 16008872.12,
 'totalOrders': 99441,
 'avgTicket': 160.58,
 'deliveryRate': 97.02,
 'avgReviewScore': 4.09,
 'totalCustomers': 96096,
 'totalSellers': 3095,
 'lateDeliveryRate': 6.77,
 'onTimeAvgScore': 4.29,
 'lateAvgScore': 2.27,
 'lateOrdersRevenue': 1150865.66,
 'reviewsWithoutComment': 58247,
 'totalReviews': 98673}

## 4. Historia 1 — Impacto de la latencia → `latency_review.json`

*Como Director de Operaciones, quiero visualizar la diferencia en días entre la fecha
estimada y la real de entrega, para identificar si el incumplimiento es el factor raíz
de las reseñas de 1–2★.*

Agrupamos las órdenes entregadas por días de atraso (positivo) o anticipación
(negativo) y por región del cliente (filtro interactivo del slide). Cada punto lleva
el score promedio y el número de órdenes.


In [5]:
lat = delivered.dropna(subset=["delay_days", "review_score", "region"]).copy()
lat["delay_days"] = lat["delay_days"].clip(-30, 30).astype(int)

def aggregate_latency(frame, region_name):
    grouped = (
        frame.groupby("delay_days", as_index=False)
        .agg(avgScore=("review_score", "mean"), orders=("order_id", "nunique"))
    )
    grouped["avgScore"] = grouped["avgScore"].round(2)
    grouped["region"] = region_name
    return grouped

frames = [aggregate_latency(lat, "Todas")]
for region_name in ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]:
    frames.append(aggregate_latency(lat[lat["region"] == region_name], region_name))

latency = pd.concat(frames, ignore_index=True).rename(columns={"delay_days": "delay"})

latency_payload = {
    "regions": ["Todas", "Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"],
    "points": latency[["region", "delay", "avgScore", "orders"]].to_dict(orient="records"),
}
(OUT / "latency_review.json").write_text(json.dumps(latency_payload, indent=2))
print(f"{len(latency_payload['points'])} puntos agregados")
latency[latency["region"] == "Todas"].query("delay in [-10, -5, 0, 5, 10, 20]")


362 puntos agregados


,delay,avgScore,orders,region
20,-10,4.33,4618,Todas
25,-5,4.19,2208,Todas
30,0,4.03,1280,Todas
35,5,2.20,434,Todas
40,10,1.58,207,Todas
50,20,1.63,86,Todas


## 5. Historia 2 — Cuantificando el riesgo → `payment_by_score.json` y `financial_kpis.json`

*Como Líder de CX, quiero identificar el volumen total de ingresos asociado a reseñas
negativas, para cuantificar el impacto monetario del mal servicio.*

Definiciones del plan v2:
- **Revenue Total** = suma global de `payment_value` (todas las órdenes).
- **Revenue en Riesgo** = suma de `payment_value` de órdenes con score 1–2
  (todas las órdenes con reseña, no solo las entregadas: una orden que nunca llegó
  también es dinero en riesgo).
- **% de Riesgo** = Revenue en Riesgo / Revenue Total × 100.

Además se exporta la evolución trimestral del Revenue en Riesgo para el selector
de trimestre del slide (se descartan trimestres con menos de 100 órdenes en los
extremos del dataset).


In [6]:
reviewed = orders_full.dropna(subset=["review_score", "payment_value"]).copy()
reviewed["review_score"] = reviewed["review_score"].astype(int)

pay_score = (
    reviewed.groupby("review_score", as_index=False)
    .agg(paymentValue=("payment_value", "sum"), orders=("order_id", "nunique"))
    .rename(columns={"review_score": "score"})
    .sort_values("score")
)
pay_score["paymentValue"] = pay_score["paymentValue"].round(2)
pay_score["pctValue"] = (pay_score["paymentValue"] / pay_score["paymentValue"].sum() * 100).round(2)

(OUT / "payment_by_score.json").write_text(json.dumps(pay_score.to_dict(orient="records"), indent=2))
pay_score


,score,paymentValue,orders,pctValue
0,1,"2,221,488.75",11364,14.01
1,2,"540,589.78",3130,3.41
2,3,"1,233,351.59",8134,7.78
3,4,"2,948,829.39",19043,18.59
4,5,"8,914,846.28",57001,56.21


In [7]:
at_risk_mask = reviewed["review_score"] <= 2
revenue_at_risk = float(reviewed.loc[at_risk_mask, "payment_value"].sum())
at_risk_orders = int(reviewed.loc[at_risk_mask, "order_id"].nunique())
risk_pct = revenue_at_risk / total_revenue * 100

# Evolucion trimestral (trimestre de compra)
paid = orders_full.dropna(subset=["payment_value"]).copy()
paid["quarter"] = (
    paid["order_purchase_timestamp"].dt.to_period("Q").astype(str).str.replace("Q", "-Q")
)
risk_paid = paid[paid["review_score"] <= 2]

by_quarter = (
    paid.groupby("quarter")
    .agg(revenue=("payment_value", "sum"), orders=("order_id", "nunique"))
    .join(risk_paid.groupby("quarter")["payment_value"].sum().rename("atRisk"))
    .fillna({"atRisk": 0.0})
    .reset_index()
    .sort_values("quarter")
)
by_quarter["riskPct"] = (by_quarter["atRisk"] / by_quarter["revenue"] * 100).round(2)
for col in ["revenue", "atRisk"]:
    by_quarter[col] = by_quarter[col].round(2)

dropped = by_quarter[by_quarter["orders"] < 100]
print(f"Trimestres descartados por volumen (<100 ordenes): {dropped['quarter'].tolist()}")
by_quarter = by_quarter[by_quarter["orders"] >= 100]

financial = {
    "total": {
        "revenue": round(total_revenue, 2),
        "atRisk": round(revenue_at_risk, 2),
        "riskPct": round(risk_pct, 2),
        "atRiskOrders": at_risk_orders,
    },
    "quarters": by_quarter.to_dict(orient="records"),
}
(OUT / "financial_kpis.json").write_text(json.dumps(financial, indent=2))

print(f"Revenue total:    {total_revenue:>15,.2f} BRL")
print(f"Revenue en riesgo:{revenue_at_risk:>15,.2f} BRL ({risk_pct:.2f}%) en {at_risk_orders:,} ordenes")
by_quarter


Trimestres descartados por volumen (<100 ordenes): ['2016-Q3', '2018-Q4']
Revenue total:      16,008,872.12 BRL
Revenue en riesgo:   2,762,078.53 BRL (17.25%) en 14,494 ordenes


,quarter,revenue,orders,atRisk,riskPct
1,2016-Q4,"59,110.10",325,"17,488.98",29.59
2,2017-Q1,"880,259.65",5262,"142,542.34",16.19
3,2017-Q2,"1,521,983.23",9349,"239,759.26",15.75
4,2017-Q3,"1,994,541.69",12642,"286,603.37",14.37
5,2017-Q4,"2,852,962.16",17848,"553,464.75",19.40
6,2018-Q1,"3,267,119.64",21208,"763,068.74",23.36
7,2018-Q2,"3,338,648.13",19979,"464,217.69",13.90
8,2018-Q3,"2,093,405.61",12820,"294,313.52",14.06


## 6. Historia 3 — Culpables y oportunidades → `category_matrix.json`

*Como Gerente Comercial, quiero identificar categorías con el peor `review_score`
promedio y mayor `freight_value`, para evaluar rentabilidad y priorizar acciones.*

Top 20 categorías por volumen con flete promedio, precio promedio y score promedio
(la tabla del slide permite ordenar por cualquier columna).


In [8]:
items_cat = (
    items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
    .merge(reviews_per_order, on="order_id", how="left")
)

category = (
    items_cat.dropna(subset=["product_category_name"])
    .groupby(["product_category_name", "product_category_name_english"], as_index=False)
    .agg(
        volume=("order_id", "count"),
        revenue=("price", "sum"),
        avgFreight=("freight_value", "mean"),
        avgPrice=("price", "mean"),
        avgReview=("review_score", "mean"),
    )
    .rename(columns={
        "product_category_name": "category",
        "product_category_name_english": "categoryEn",
    })
    .sort_values("volume", ascending=False)
    .head(20)
)
for col in ["revenue", "avgFreight", "avgPrice", "avgReview"]:
    category[col] = category[col].round(2)

(OUT / "category_matrix.json").write_text(
    json.dumps(category.to_dict(orient="records"), indent=2, ensure_ascii=False)
)
category.head(10)


,category,categoryEn,volume,revenue,avgFreight,avgPrice,avgReview
13,cama_mesa_banho,bed_bath_table,11115,"1,036,988.68",18.42,93.30,3.90
11,beleza_saude,health_beauty,9670,"1,258,681.34",18.88,130.16,4.14
32,esporte_lazer,sports_leisure,8641,"988,048.97",19.51,114.34,4.11
54,moveis_decoracao,furniture_decor,8334,"729,762.49",20.73,87.56,3.90
44,informatica_acessorios,computers_accessories,7827,"911,954.32",18.82,116.51,3.93
70,utilidades_domesticas,housewares,6964,"632,248.66",20.99,90.79,4.05
64,relogios_presentes,watches_gifts,5991,"1,205,005.68",16.78,201.14,4.02
68,telefonia,telephony,4545,"323,667.53",15.67,71.21,3.95
40,ferramentas_jardim,garden_tools,4347,"485,256.46",22.77,111.63,4.05
8,automotivo,auto,4235,"592,720.11",21.88,139.96,4.06


## 7. Validación y resumen

Consistencia interna de las cifras exportadas y lista de archivos generados.


In [9]:
# El revenue total debe ser exactamente la suma de payments
assert abs(total_revenue - float(payments["payment_value"].sum())) < 1e-6

# La distribucion por score debe sumar el total de ordenes con resena y pago
assert abs(float(pay_score["paymentValue"].sum()) - float(reviewed["payment_value"].sum())) < 1.0

# El revenue en riesgo debe coincidir con los segmentos 1-2 del donut
donut_risk = float(pay_score.loc[pay_score["score"] <= 2, "paymentValue"].sum())
assert abs(donut_risk - revenue_at_risk) < 1.0

print("Cifras clave de la presentacion:")
print(f"  Revenue total:        {total_revenue:,.2f} BRL")
print(f"  Revenue en riesgo:    {revenue_at_risk:,.2f} BRL ({risk_pct:.2f}%)")
print(f"  Ordenes 1-2 estrellas: {at_risk_orders:,}")
print(f"  Entregas tardias:     {kpis['lateDeliveryRate']}% | score a tiempo {kpis['onTimeAvgScore']} vs tardio {kpis['lateAvgScore']}")
print(f"  Resenas sin comentario: {kpis['reviewsWithoutComment']:,}")
print()
print("Archivos generados en src/data/:")
for f in sorted(OUT.glob("*.json")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")


Cifras clave de la presentacion:
  Revenue total:        16,008,872.12 BRL
  Revenue en riesgo:    2,762,078.53 BRL (17.25%)
  Ordenes 1-2 estrellas: 14,494
  Entregas tardias:     6.77% | score a tiempo 4.29 vs tardio 2.27
  Resenas sin comentario: 58,247

Archivos generados en src/data/:
  category_matrix.json (3,862 bytes)
  financial_kpis.json (1,275 bytes)
  kpis.json (354 bytes)
  latency_review.json (37,097 bytes)
  payment_by_score.json (497 bytes)
